This script calculates density of various point datasets along the road network. The input is the road network downloaded from OSM. The output is the .gpkg file identical to the input roads file, with the new columns added representing: count, kde, and rank of points, by point type.

Then these columns are normalised 0-1 to be used in composite indicators. The indicators generated with this notebook are:

<!-- - People choose to walk, cycle and use public transport -->
<!-- - Pedestrians from all walks of life -->
<!-- - Easy to cross -->
- People feel safe
- Things to see and do
- Not too noisy
- Clean air
- Shade and shelter   
<!-- - Places to stop and rest -->
<!-- - People feel relaxed -->

All datasets were clipped to Greater London boundary before being used in this notebook.

In [5]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from shapely.geometry import Point, LineString, MultiLineString

# These are the custom scripts written for this research piece
#__________________________________________________________________________________________________
from network_kde import network_kde_on_lines
# This script computes line-based kernel density estimates (KDE) by projecting point events onto nearby 
# line segments. It is designed for street-network style analysis where points represent observations or 
# amenities, and lines represent street segments or other linear features.

from transfer_point_values_to_roads_by_matched_name import transfer_point_values_to_roads_by_matched_name
#This function transfers point-based values onto roads using a nearest-road match and then aggregates 
#those values with the median. The reason to calculate these points differently is that traffic count points 
#are sparsely located, but the traffic moves past the count point, so it is fair to say that the street as a 
#whole has a chance to have similar traffic counts in other locations, not just in the street section 
#where there has been a survey done. The roads are matched by name

In [2]:
# import matplotlib.pyplot as plt

In [3]:
# little function to define the file root on different machines
def find_qm_root(start_path: Path = Path.cwd(), anchor: str = "CASA0025_Project") -> Path:
    """
    Traverse up from the start_path until the anchor folder is found. Returns the path to the anchor folder.
    Anchor folder in this script is the Github repository name
    """
    for parent in [start_path] + list(start_path.parents):
        if parent.name == anchor:
            return parent
    raise FileNotFoundError(f"Anchor folder '{anchor}' not found in path hierarchy.")
  
qm_root = find_qm_root()

In [3]:
#Load the initial file with roads
roads_gdf = gpd.read_file(qm_root / "composite_indicators_module/data/road_lines/260422_network_routing_input.gpkg")

In [4]:
#Filter the selected columns
roads_gdf = roads_gdf[['osm_id', 'name','geometry']].copy() #smaller subset for computing

# People feel safe - Crime

This section computes the point density of indicators related to safety from crime

In [5]:
crime_gdf = gpd.read_file(qm_root / "composite_indicators_module/data/point_data/crime/crime_points.gpkg")

In [6]:
roads_crime = network_kde_on_lines(
    point_gdf=crime_gdf,
    line_gdf=roads_gdf,
    value_name="crime",
    mode="all_only",
    bandwidth=50,
    kernel="gaussian",
    target_crs="EPSG:27700",
    min_segment_length=90
)

Processed 1000 eligible line segments...
Processed 2000 eligible line segments...
Processed 3000 eligible line segments...
Processed 4000 eligible line segments...
Processed 7000 eligible line segments...
Processed 8000 eligible line segments...
Processed 9000 eligible line segments...
Processed 10000 eligible line segments...
Processed 11000 eligible line segments...
Processed 12000 eligible line segments...
Processed 14000 eligible line segments...
Processed 16000 eligible line segments...
Processed 19000 eligible line segments...
Processed 22000 eligible line segments...
Processed 24000 eligible line segments...
Processed 25000 eligible line segments...
Processed 26000 eligible line segments...
Processed 27000 eligible line segments...
Processed 29000 eligible line segments...
Processed 31000 eligible line segments...


In [7]:
#uncomment this if th kernel breaks - this line will create a backup copy of the file after this step
roads_crime.to_file("roads_crime.gpkg", driver="GPKG") 

In [8]:
del crime_gdf #clean the memory after computing

# Things to see and do - Historic buildings

This section computes the point density of listed buildings

In [9]:
listed_gdf = gpd.read_file(qm_root / "composite_indicators_module/data/point_data/listed_b/listed_b_london.gpkg")

In [10]:
roads_crime_listed = network_kde_on_lines(
    point_gdf=listed_gdf,
    line_gdf=roads_crime,
    value_name="history",
    mode="all_only",
    bandwidth=100,
    kernel="gaussian",
    target_crs="EPSG:27700",
    min_segment_length=90
)

Processed 2000 eligible line segments...
Processed 3000 eligible line segments...
Processed 6000 eligible line segments...
Processed 17000 eligible line segments...
Processed 29000 eligible line segments...


In [11]:
#uncomment this if th kernel breaks - this line will create a backup copy of the file after this step
roads_crime_listed.to_file("roads_crime_listed.gpkg", driver="GPKG")

In [12]:
del listed_gdf

# People feel safe - Traffic

This section computes the point density of pedestrian collisions

In [13]:
collisions_gdf = gpd.read_file(qm_root / "composite_indicators_module/data/point_data/collisions/collisions_2026_2023.gpkg")

In [14]:
roads_crime_listed_collisions = network_kde_on_lines(
    point_gdf=collisions_gdf,
    line_gdf=roads_crime_listed,
    value_name="ped_collisions",
    mode="all_only",
    bandwidth=100,
    kernel="gaussian",
    target_crs="EPSG:27700",
    min_segment_length=90
)

Processed 1000 eligible line segments...
Processed 2000 eligible line segments...
Processed 4000 eligible line segments...
Processed 9000 eligible line segments...
Processed 10000 eligible line segments...
Processed 11000 eligible line segments...
Processed 16000 eligible line segments...
Processed 18000 eligible line segments...
Processed 19000 eligible line segments...
Processed 31000 eligible line segments...


In [15]:
#uncomment this if th kernel breaks - this line will create a backup copy of the file after this step
roads_crime_listed_collisions.to_file("roads_crime_listed_collisions.gpkg", driver="GPKG")

In [16]:
del collisions_gdf

# Things to see and do - General points of interest

This section computes the point density of POI

In [17]:
poi_gdf = gpd.read_file(qm_root / "composite_indicators_module/data/point_data/poi/data/poi_london.gpkg")

In [18]:
roads_crime_listed_collisions_poi = network_kde_on_lines(
    point_gdf=poi_gdf,
    line_gdf=roads_crime_listed_collisions,
    value_name="poi",
    mode="all_only",
    bandwidth=100,
    kernel="gaussian",
    target_crs="EPSG:27700",
    min_segment_length=90
)

Processed 1000 eligible line segments...
Processed 2000 eligible line segments...
Processed 3000 eligible line segments...
Processed 4000 eligible line segments...
Processed 5000 eligible line segments...
Processed 7000 eligible line segments...
Processed 8000 eligible line segments...
Processed 9000 eligible line segments...
Processed 10000 eligible line segments...
Processed 11000 eligible line segments...
Processed 12000 eligible line segments...
Processed 14000 eligible line segments...
Processed 15000 eligible line segments...
Processed 16000 eligible line segments...
Processed 18000 eligible line segments...
Processed 19000 eligible line segments...
Processed 22000 eligible line segments...
Processed 23000 eligible line segments...
Processed 24000 eligible line segments...
Processed 25000 eligible line segments...
Processed 26000 eligible line segments...
Processed 27000 eligible line segments...
Processed 28000 eligible line segments...
Processed 29000 eligible line segments...


In [19]:
#uncomment this if th kernel breaks - this line will create a backup copy of the file after this step
roads_crime_listed_collisions_poi.to_file("roads_crime_listed_collisions_poi.gpkg", driver="GPKG")

In [20]:
del poi_gdf

# Not too noisy % Clean Air - Traffic counts

This section computes the median traffic counts on the street

In [3]:
# roads_crime_listed_collisions_poi = gpd.read_file(qm_root / "composite_indicators_module/roads_crime_listed_collisions_poi.gpkg")

In [4]:
traffic_gdf = gpd.read_file(qm_root / "composite_indicators_module/data/point_data/traffic/traffic_2023_2026.gpkg")

In [5]:
roads_crime_listed_collisions_poi_traffic = transfer_point_values_to_roads_by_matched_name(
    point_gdf=traffic_gdf,
    road_gdf=roads_crime_listed_collisions_poi,
    road_name_col="name",
    point_value_cols=["all_motor_vehicles", "all_HGVs"],
    target_crs="EPSG:27700",
    max_match_distance=100,
    prefix="count_"
)

In [7]:
roads_crime_listed_collisions_poi_traffic.to_file("roads_crime_listed_collisions_poi_traffic.gpkg", driver="GPKG")

# Integrating the GEE-originated columns

This section exports a super-light .shp file for Google Earth Engine, and loads back the results of the batch computation

In [4]:
roads_with_points = gpd.read_file(qm_root / "composite_indicators_module/roads_crime_listed_collisions_poi_traffic.gpkg")

In [5]:
# assign a unique id for joining
# osmid doesn't work as there are a lot of duplicated ids
roads_with_points['our_uid'] = range(len(roads_with_points))

In [16]:
#This code block downloads a light version of the geodataframe to be used in GEE as an asset for batch request
#It is to be done only once

# roads_export_1 = roads_with_points[
#     ['our_uid', 'geometry']
# ].to_crs(epsg=3857)

# roads_export_1.to_file(
#     "data/GEE_input/export_for_raster_processing.shp",
#     driver="ESRI Shapefile"
# )

In [9]:
# GEE batch export is a series of csv files

shade_folder = (qm_root / "composite_indicators_module/data/GEE_output/shade_shelter")
shade_csv_files = sorted(shade_folder.glob("*.csv"))
shade_df = pd.concat((pd.read_csv(f) for f in shade_csv_files), ignore_index=True)

In [10]:
print(shade_df.shape)
shade_df.head()

(1942494, 5)


,our_uid,shadow,surface_temp,wind_speed,wind_protected
0,69344,0.115278,15647.0,3.349988,0.033510
1,56850,0.283786,15597.0,3.089419,0.048833
2,62350,0.013427,15735.0,3.119959,0.017910
3,62352,0.011567,15735.0,3.119959,0.007938
4,62351,0.012947,15735.0,3.119959,0.012539


In [11]:
shade_df[shade_df['shadow'].isnull()]

,our_uid,shadow,surface_temp,wind_speed,wind_protected
97,26174,NaN,NaN,NaN,NaN
291,59121,NaN,NaN,NaN,NaN
802,26167,NaN,NaN,NaN,NaN
1237,59088,NaN,NaN,NaN,NaN
1279,26170,NaN,NaN,NaN,NaN
...,...,...,...,...,...
1942394,986003,NaN,NaN,NaN,NaN
1942417,976758,NaN,NaN,NaN,NaN
1942427,976727,NaN,NaN,NaN,NaN
1942442,972692,NaN,NaN,NaN,NaN


In [9]:
roads_with_points_shade_wind = roads_with_points.merge(shade_df, on="our_uid", how="left")

In [12]:
roads_with_points_shade_wind.to_file("roads_with_points_shade_wind.gpkg", driver="GPKG")

# Normalisation and Indicator building 

This section computes the point density of indicators related to safety from crime

In [4]:
roads_with_points_shade_wind = gpd.read_file(qm_root / "composite_indicators_module/roads_with_points_shade_wind.gpkg")

In [6]:
roads_with_points_shade_wind.columns

Index(['osm_id', 'name', 'segment_length', 'kde_eligible', 'count_crime',
       'kde_crime', 'rank_crime', 'count_history', 'kde_history',
       'rank_history', 'count_ped_collisions', 'kde_ped_collisions',
       'rank_ped_collisions', 'count_poi', 'kde_poi', 'rank_poi',
       '_road_seg_id', '_road_name_clean', '_group_key',
       'count_all_motor_vehicles_med', 'count_all_hgvs_med', 'count_n_points',
       'our_uid', 'shadow', 'surface_temp', 'wind_speed', 'wind_protected',
       'geometry'],
      dtype='object')

In [7]:
# roads_with_points_shade_wind[
#     ['kde_poi', 'kde_crime', 'kde_history', 'kde_ped_collisions']
# ].describe()

In [9]:
kde_cols = [
    'kde_poi',
    'kde_crime',
    'kde_history',
    'kde_ped_collisions'
]

kde_max = 0.2  # chosen saturation threshold

for col in kde_cols:
    # Clip values to [0, kde_max]
    roads_with_points_shade_wind[col] = (
        roads_with_points_shade_wind[col]
        .clip(0, kde_max)
    )

    # Create scaled version in [0, 1]
    roads_with_points_shade_wind[f'{col}'] = (
        roads_with_points_shade_wind[col] / kde_max
    )

In [10]:
# imputing mean values for NULLs
for col in ["wind_speed", "surface_temp",
           'shadow', 'wind_protected']: #imputing also the % cover, because our roads layer included lines outside london
    roads_with_points_shade_wind[col] = (
        roads_with_points_shade_wind[col]
        .fillna(roads_with_points_shade_wind[col].mean())
    )

In [11]:
cols = ['kde_crime', 'kde_history','kde_ped_collisions','kde_poi',
       'count_all_motor_vehicles_med', 'count_all_hgvs_med', 'count_n_points', #not being used at this iteration
       'shadow', 'surface_temp', 'wind_speed', 'wind_protected']

In [12]:
scaler = MinMaxScaler() #scaling 0-1
roads_with_points_shade_wind[cols] = scaler.fit_transform(roads_with_points_shade_wind[cols])

In [15]:
# Things to see and do - poi + listed buildings, equal weights. Positive indicator
roads_with_points_shade_wind['score_things_see_do'] = roads_with_points_shade_wind['kde_poi'] + roads_with_points_shade_wind['kde_history']
roads_with_points_shade_wind['score_things_see_do'] = scaler.fit_transform(
    roads_with_points_shade_wind[['score_things_see_do']]
)

In [16]:
# Feeling safe - crime points + pedestrian collisions, equal weight. Negative indicator
roads_with_points_shade_wind['score_feel_safe'] = roads_with_points_shade_wind['kde_crime'] + roads_with_points_shade_wind['kde_ped_collisions']
roads_with_points_shade_wind['score_feel_safe'] = scaler.fit_transform(
    roads_with_points_shade_wind[['score_feel_safe']]
)

In [18]:
# Shade and shelter - shadow cover + wind cover - wind speed (higher is worse) - temperature (higher is worse), equal weight. Positive indicator
roads_with_points_shade_wind['score_shade_shelter'] = roads_with_points_shade_wind['shadow'] + roads_with_points_shade_wind['wind_protected'] - roads_with_points_shade_wind['wind_speed'] - roads_with_points_shade_wind['surface_temp']
roads_with_points_shade_wind['score_shade_shelter'] = scaler.fit_transform(
    roads_with_points_shade_wind[['score_shade_shelter']]
)

In [19]:
# This line below exports the full roads gpkg with all columns, after the point-based 
# data was added, and after shade and shelter from GEE was added
roads_with_points_shade_wind.to_file("roads_with_points_shade_wind_scaled.gpkg", driver="GPKG")

In [20]:
#THIS IS WHERE THE OTHER COLUMNS SHOULD BE ADDED IN, IF WE DECIDE TO ADD MORE DATA INTO THE MIX

In [21]:
#Final export with selected columns
roads_export_scored = roads_with_points_shade_wind[
    ['our_uid', 'osm_id', 'name', 'score_things_see_do', 'score_feel_safe','score_shade_shelter','geometry']
].to_crs(epsg=3857)

roads_export_scored.to_file(
    "260422_roads_export_final.gpkg",
    driver="GPKG"
)